<a href="https://colab.research.google.com/github/CienciaDatosUdea/005_CCA_Estudiantes/blob/main/Laboratorios/03_Lab_naive_bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Laboratorio: Naive Bayes Bernoulli para detectar spam

## Objetivo
Construir desde cero un clasificador **Naive Bayes Bernoulli** a partir de un corpus pequeño de correos.

Al terminar debes poder explicar y calcular:

$
P(Y),\qquad P(X_i\mid Y),\qquad P(X\mid Y),\qquad P(Y\mid X)
$

y comprender por qué la hipótesis

$
X_i \perp X_j\mid Y
$

reduce drásticamente la complejidad del modelo.

## Contexto

Queremos clasificar un correo como

$
Y=1:\ \text{spam}, \qquad Y=0:\ \text{normal}.
$

Usaremos cinco palabras del vocabulario:


$V=\{\text{dinero, gratis, premio, proyecto, reunion}\}.$

Cada correo se representa por

$X=(X_1,\ldots,X_5),$

donde

$
X_i=\begin{cases}
1 & \text{si aparece la palabra }i,\\
0 & \text{si no aparece.}
\end{cases}
$
Por ejemplo, "dinero gratis" se representa como $(1,1,0,0,0).$


In [1]:
corpus = [
    ("spam",   "gana dinero gratis"),
    ("spam",   "dinero gratis ahora"),
    ("spam",   "premio dinero gratis"),
    ("spam",   "gana premio ahora"),
    ("spam",   "en la reunion habra dinero gratis"),
    ("normal",  "daremos un premio despues de la reunion"),
    ("normal", "reunion de proyecto"),
    ("normal", "proyecto para mañana"),
    ("normal", "reunion mañana"),
    ("normal", "informe del proyecto"),
    ("normal", "informe del proyecto"),
]

vocabulario = ["dinero", "gratis", "premio", "proyecto", "reunion"]


## Parte 1 — Construir el vector \(X\)

1. Escribe una función `vectorizar(texto, vocabulario)` que transforme un correo en un vector binario.
2. Vectoriza todos los correos del corpus.
3. Verifica manualmente al menos dos ejemplos.

Ejemplo esperado:


$\text{"gana dinero gratis"}\longrightarrow(1,1,0,0,0)$


In [2]:
# TODO: implementa vectorizar(texto, vocabulario)
import numpy as np
x = np.array([1, 1, 1, 1, 1])

In [3]:
# 1. Escribe una función `vectorizar(texto, vocabulario)` que transforme un correo en un vector binario.

def vectorizar(texto, vocabulario):
    texto = texto.lower() #estandarizar mayúsculas y minúsculas
    vector = np.zeros(len(vocabulario), dtype=int) 
    for i in range(len(vocabulario)):
        palabra = vocabulario[i]
        if palabra in texto:
            vector[i] = 1
    return vector
         
    
    
a=vectorizar("gana dinero gratis", vocabulario)
print(a)

[1 1 0 0 0]


In [4]:
# 2. Vectoriza todos los correos del corpus.
Y = [] # Lista para etiquetas
X = [] # Lista para texto

for i in range(len(corpus)):
    etiqueta = corpus[i][0]
    texto    = corpus[i][1]
    vector   = vectorizar(texto, vocabulario)
    Y.append(etiqueta)
    X.append(vector)

print(np.array(Y))
print(np.array(X))

['spam' 'spam' 'spam' 'spam' 'spam' 'normal' 'normal' 'normal' 'normal'
 'normal' 'normal']
[[1 1 0 0 0]
 [1 1 0 0 0]
 [1 1 1 0 0]
 [0 0 1 0 0]
 [1 1 0 0 1]
 [0 0 1 0 1]
 [0 0 0 1 1]
 [0 0 0 1 0]
 [0 0 0 0 1]
 [0 0 0 1 0]
 [0 0 0 1 0]]


In [5]:
# Verificar manualmente 2 ejemplos
ejm1 = vectorizar(corpus[0][1], vocabulario)
ejm2 = vectorizar(corpus[10][1], vocabulario)

print(ejm1)
print(ejm2)


[1 1 0 0 0]
[0 0 0 1 0]


## Parte 2 — Calcular el prior \(P(Y)\)

Calcula:


$P(Y=\text{spam}),\qquad P(Y=\text{normal}).$


Recuerda:

$
P(Y=y)=\frac{\#\text{correos de clase }y}{\#\text{correos totales}}.
$

In [6]:
# TODO: calcula P(Y=spam) 
totales = len(Y)
spam = Y.count('spam')
normal = Y.count('normal')
P_spam = spam / totales
P_normal = normal / totales

print("La probabilidad de que un correo sea spam es:", P_spam)
print("La probabilidad de que un correo sea normal es:", P_normal)


La probabilidad de que un correo sea spam es: 0.45454545454545453
La probabilidad de que un correo sea normal es: 0.5454545454545454


## Parte 3 — Calcular $P(X_i=1\mid Y)$

Para cada palabra calcula su frecuencia dentro de cada clase. Por ejemplo:

$
P(X_{dinero}=1\mid Y=spam)
=\frac{\#\text{spam que contienen dinero}}{\#\text{spam}}.
$

Construye una tabla con las cinco palabras y ambas clases.

**Pregunta:** ¿qué ocurre si una palabra nunca aparece en una clase?

In [7]:
# TODO: calcula las probabilidades condicionales sin suavizado

etiquetas = ["spam", "normal"]
def calcular_probabilidades_condicionales(X, Y, vocabulario, etiquetas):
    tabla = {}
    
    for etiqueta in etiquetas:
        conteos = np.zeros(len(vocabulario))
        total_etiqueta = 0  
        
        for i in range(len(Y)): # recorrer sobre los 11 correos
            if Y[i] == etiqueta:
                total_etiqueta += 1
                for j in range(len(vocabulario)): # recorrer sobre las 5 palabras del vocabulario
                    if X[i][j] == 1:
                        conteos[j] += 1
                        
        probabilidades = conteos / total_etiqueta
        tabla[etiqueta] = probabilidades
        
    return tabla


In [8]:
for etiqueta in etiquetas:
    print(f"Probabilidades condicionales para {etiqueta}:")
    for j in range(len(vocabulario)):
        print(f"P({vocabulario[j]}|{etiqueta}) = {calcular_probabilidades_condicionales(X, Y, vocabulario, etiquetas)[etiqueta][j]}")

Probabilidades condicionales para spam:
P(dinero|spam) = 0.8
P(gratis|spam) = 0.8
P(premio|spam) = 0.4
P(proyecto|spam) = 0.0
P(reunion|spam) = 0.2
Probabilidades condicionales para normal:
P(dinero|normal) = 0.0
P(gratis|normal) = 0.0
P(premio|normal) = 0.16666666666666666
P(proyecto|normal) = 0.6666666666666666
P(reunion|normal) = 0.5


## Parte 4 — Suavizado de Laplace

Para evitar probabilidades exactamente iguales a cero, usa suavizado de Laplace para variables Bernoulli:


$\hat P(X_i=1\mid Y=y)=\frac{N_{iy}+1}{N_y+2},$


donde \(N_{iy}\) es el número de correos de clase \(y\) que contienen la palabra \(i\), y \(N_y\) es el número total de correos de esa clase.

Calcula de nuevo la tabla.

In [9]:
# TODO: calcula probabilidades condicionales con Laplace
def calcular_probabilidades_condicionales_laplace(X, Y, vocabulario, etiquetas):
    tabla_laplace = {}
    
    for etiqueta in etiquetas:
        conteos = np.zeros(len(vocabulario))
        total_etiqueta = 0  
        
        for i in range(len(Y)): # recorrer sobre los 11 correos
            if Y[i] == etiqueta:
                total_etiqueta += 1
                for j in range(len(vocabulario)): # recorrer sobre las 5 palabras del vocabulario
                    if X[i][j] == 1:
                        conteos[j] += 1
                        
        probabilidades = (conteos + 1) / (total_etiqueta + 2)
        tabla_laplace[etiqueta] = probabilidades
        
    return tabla_laplace

In [10]:
for etiqueta in etiquetas:
    print(f"Probabilidades condicionales con Laplace para {etiqueta}:")
    for j in range(len(vocabulario)):
        print(f"P({vocabulario[j]}|{etiqueta}) = {calcular_probabilidades_condicionales_laplace(X, Y, vocabulario, etiquetas)[etiqueta][j]}")
        
# Esto da la probabilidad de presencia

Probabilidades condicionales con Laplace para spam:
P(dinero|spam) = 0.7142857142857143
P(gratis|spam) = 0.7142857142857143
P(premio|spam) = 0.42857142857142855
P(proyecto|spam) = 0.14285714285714285
P(reunion|spam) = 0.2857142857142857
Probabilidades condicionales con Laplace para normal:
P(dinero|normal) = 0.125
P(gratis|normal) = 0.125
P(premio|normal) = 0.25
P(proyecto|normal) = 0.625
P(reunion|normal) = 0.5


## Parte 5 — Clasificar un correo nuevo

Clasifica:

> **"dinero gratis"**

Su vector es

$x=(1,1,0,0,0).$

Bajo Naive Bayes Bernoulli:

$P(x\mid Y=y)=\prod_iP(X_i=x_i\mid Y=y).$

Recuerda que para una palabra ausente:

$P(X_i=0\mid Y=y)=1-P(X_i=1\mid Y=y).$


Calcula los scores conjuntos:

$S_y=P(Y=y)P(x\mid Y=y),$

y finalmente:

$P(Y=y\mid x)=\frac{S_y}{S_{spam}+S_{normal}}.$

Decide la clase del correo.

In [11]:
tabla_laplace = calcular_probabilidades_condicionales_laplace(X, Y, vocabulario, etiquetas)

print(tabla_laplace)

{'spam': array([0.71428571, 0.71428571, 0.42857143, 0.14285714, 0.28571429]), 'normal': array([0.125, 0.125, 0.25 , 0.625, 0.5  ])}


In [12]:
# TODO: como computar likelihood, score conjunto y posterior para "dinero gratis"

# 1. Vectorizar el correo "dinero gratis"
x = vectorizar("dinero gratis", vocabulario)

# 2. Calcular la probabilidad usando la independencia condicional: si ya sabemos que es spam, las palabras son independientes entre sí.

def probabilidad_x_dado_etiqueta(x, probabilidades_clase):
    p = 1.0
    for j in range(len(x)):
        if x[j] == 1:
            p = p * probabilidades_clase[j]
        else:
            p = p * (1 - probabilidades_clase[j])
    return p

# 3. Calcular el score conjunto para cada clase

x = vectorizar("dinero gratis", vocabulario)

P_x_dado_spam   = probabilidad_x_dado_etiqueta(x, tabla_laplace["spam"])
P_x_dado_normal = probabilidad_x_dado_etiqueta(x, tabla_laplace["normal"])

S_spam   = P_spam   * P_x_dado_spam
S_normal = P_normal * P_x_dado_normal

# 4. Calcular el posterior para cada clase
P_spam_dado_x   = S_spam   / (S_spam + S_normal)
P_normal_dado_x = S_normal / (S_spam + S_normal)

In [13]:
print("P(spam|x) =", P_spam_dado_x)
print("P(normal|x) =", P_normal_dado_x)
print("P(x|spam) =", P_x_dado_spam)
print("P(x|normal) =", P_x_dado_normal)
print("S_spam =", S_spam)
print("S_normal =", S_normal)

P(spam|x) = 0.9854432517009721
P(normal|x) = 0.014556748299027745
P(x|spam) = 0.17849705479859584
P(x|normal) = 0.002197265625
S_spam = 0.08113502490845266
S_normal = 0.0011985085227272725


In [15]:
if P_spam_dado_x > P_normal_dado_x:
    print("El correo ""dinero gratis"" se clasifica como SPAM")
else:
    print("El correo ""dinero gratis"" se clasifica como NORMAL")
    

El correo dinero gratis se clasifica como SPAM


## Parte 6 — Interpretación

Responde brevemente:

1. ¿Dónde se usa la hipótesis de independencia condicional?
2. ¿Por qué no necesitamos almacenar una probabilidad para cada uno de los \(2^5\) vectores posibles?
3. ¿Por qué Naive Bayes se considera un modelo **generativo** aunque aquí lo usemos para clasificar?

## Respuestas

1. ¿Dónde se usa la hipótesis de independencia condicional?

La hipótesis se usa en la función definida probabilidad_x_dado_etiqueta debido a que si ya sabemos que es spam, las palabras pueden considerarse como independientes entre sí. Por tanto, se considera:


$$P(x|Y=y)=\prod_{i=1}^{5}P(X_{i}=x_{i}|Y=y)$$

2. ¿Por qué no necesitamos almacenar una probabilidad para cada uno de los \(2^5\) vectores posibles?

Una vez determinado el correo como spam o no spam, las palabras pueden considerarse como independientes entre si, por tanto, no sería necesario considerar $2^{5} = 32$ combinaciones posibles. En su lugar, solo se necesitan 5 parámetros por clase.

3. ¿Por qué Naive Bayes se considera un modelo **generativo** aunque aquí lo usemos para clasificar?

Un modelo discriminativo aprende a calcular la probabilidad de un evento dado otro sin modelar como se distribuyen los datos y solo encuentra la frontera que mejor separa las clases. Este tipo de modelo, no puede generar datos nuevos. Por otro lado, un modelo generativo aprende la distribución conjunta, es decir, aprende cómo se ven los datos dentro de cada clase. Esto le permite calcular la probabilidad de un evento dado otro, y también generar ejemplos sintéticos nuevos.

En este caso, Naive Bayes es un modelo generativo porque no aprende P(Y∣X) directamente, sino que aprende P(Y), la frecuencia de cada clase, y P(X∣Y), la probabilidad de que aparezca cada palabra, condicionada a la clase. Con esto, se modela cómo se distribuyen las palabras dentro de los correos spam y dentro de los correos normales por separado, lo que permitiría generar correos sintéticos de cada clase.

La probabilidad que se requiere para clasificar P(Y∣X), no se aprende de forma directa, sino que se obtiene aplicando el Teorema de Bayes sobre P(Y) y P(X∣Y). Esto es distinto de lo que haría un modelo discriminativo, como la regresión logística, que ajustaría directamente P(Y∣X) sin pasar por modelar cómo se ven los datos dentro de cada clase.